# Kaggle Loading

Ноутбук надо запускать локально, чтобы загрузить папку в Kaggle Dataset через `kagglehub`. Загружаем указанные эксперименты из `experiment_runs`: чекпойнты, конфиги, токенизаторы и результаты с сохранением вложенности. Для другой папки поменяй `SOURCE_DIR`, а имена экспериментов перечисли в `EXPERIMENTS`.

В настройках укажи свой Kaggle username. Выполняй ячейки по порядку: сама загрузка начинается в ячейке `dataset_upload`. Новый датасет создаётся приватным; если он уже существует, создаётся новая версия с его текущими настройками доступа.

[Документация KaggleHub](https://github.com/Kaggle/kagglehub#upload-dataset).

In [1]:
%pip install -U kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

# Находим корень из utilities/ или любой другой папки внутри проекта.
current_dir = Path.cwd().resolve()
PROJECT_DIR = next(
    (
        path for path in (current_dir, *current_dir.parents)
        if (path / "pyproject.toml").is_file() and (path / "lib").is_dir()
    ),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Открой ноутбук внутри папки проекта HW-2.")

print(f"Корень проекта: {PROJECT_DIR}")

Корень проекта: /Users/maks62wip/ML/NLP/HW-2


## Настройки

`DATASET_HANDLE` имеет вид `username/dataset-name`. `SOURCE_DIR` — папка с экспериментами;
`EXPERIMENTS` — список имён папок, которые нужно отправить; `VERSION_NOTES` — описание обновления.

По умолчанию выбран только `baseline_plus_plus_kaggle`. Чтобы добавить другие эксперименты,
допиши их имена в список `EXPERIMENTS`.

In [3]:
DATASET_HANDLE = "wwip62/nlp-hw2-runs"
SOURCE_DIR = PROJECT_DIR / "experiment_runs"

EXPERIMENTS = ["baseline_plus_plus_longer"]

VERSION_NOTES = "Experiment runs with checkpoints, configs and tokenizers"

## Проверка выбранных экспериментов

Показываем файлы только из `EXPERIMENTS` и их общий размер. Чекпойнты в память не загружаем.
Пустой список или неизвестное имя эксперимента вызывают ошибку до отправки.

In [4]:
def prepare_upload(source_dir, experiments):
    source_dir = Path(source_dir).expanduser().resolve()
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Папка не найдена: {source_dir}")
    if not experiments:
        raise ValueError("Укажи хотя бы один эксперимент в EXPERIMENTS.")

    selected = set(experiments)
    available = {path.name for path in source_dir.iterdir() if path.is_dir()}
    missing = selected - available
    if missing:
        raise FileNotFoundError(f"Эксперименты не найдены: {', '.join(sorted(missing))}")

    files = sorted(
        path
        for name in selected
        for path in (source_dir / name).rglob("*")
        if path.is_file()
    )
    if not files:
        raise ValueError("В выбранных экспериментах нет файлов.")

    # Исключаем остальные эксперименты и файлы в корне, сохраняя вложенность выбранных.
    ignore_patterns = [
        f"{path.name}/" if path.is_dir() else path.name
        for path in source_dir.iterdir()
        if path.name not in selected
    ]
    return source_dir, files, ignore_patterns


SOURCE_DIR, files, ignore_patterns = prepare_upload(SOURCE_DIR, EXPERIMENTS)
total_bytes = sum(path.stat().st_size for path in files)
print(f"Папка: {SOURCE_DIR}")
print(f"Датасет: {DATASET_HANDLE}")
print(f"Эксперименты: {', '.join(EXPERIMENTS)}")
print(f"Файлов к загрузке: {len(files)} | размер: {total_bytes / 1024**3:.2f} GiB\n")
for path in files:
    print(f"{path.relative_to(SOURCE_DIR)} ({path.stat().st_size / 1024**2:.2f} MiB)")

Папка: /Users/maks62wip/ML/NLP/HW-2/experiment_runs
Датасет: wwip62/nlp-hw2-runs
Эксперименты: baseline_plus_plus_longer
Файлов к загрузке: 15 | размер: 1.25 GiB

baseline_plus_plus_longer/checkpoints/best.pt (637.57 MiB)
baseline_plus_plus_longer/checkpoints/last.pt (637.57 MiB)
baseline_plus_plus_longer/config.yaml (0.00 MiB)
baseline_plus_plus_longer/init_state/sample_indices.json (0.00 MiB)
baseline_plus_plus_longer/init_state/tokenizer.json (1.07 MiB)
baseline_plus_plus_longer/init_state/wandb.json (0.00 MiB)
baseline_plus_plus_longer/metrics/epochs_history.json (0.00 MiB)
baseline_plus_plus_longer/metrics/examples_history.json (0.19 MiB)
baseline_plus_plus_longer/results/summary.json (0.00 MiB)
baseline_plus_plus_longer/wandb/run-20261022_222020-2424xg38/files/output.log (0.06 MiB)
baseline_plus_plus_longer/wandb/run-20261022_222020-2424xg38/files/requirements.txt (0.02 MiB)
baseline_plus_plus_longer/wandb/run-20261022_222020-2424xg38/files/wandb-metadata.json (0.00 MiB)
baseline

## Вход в Kaggle

Надо создать API-токен в [настройках Kaggle](https://www.kaggle.com/settings/api) и введи его в запрос `login()`. Если Kaggle API уже настроен, эту ячейку можно пропустить.

In [5]:
# import kagglehub

# kagglehub.login()

## Загрузка выбранных экспериментов

Отправляем только папки из `EXPERIMENTS`. Остальные папки и отдельные файлы в корне
`SOURCE_DIR` исключаются через `ignore_patterns`; копировать чекпойнты во временную папку не нужно.

В корне датасета сохраняются имена выбранных экспериментов, например `baseline_plus_plus_kaggle/`.
Повторный запуск этой ячейки создаёт новую версию датасета с текущим набором экспериментов.

In [ ]:
import kagglehub

SOURCE_DIR, files, ignore_patterns = prepare_upload(SOURCE_DIR, EXPERIMENTS)

kagglehub.dataset_upload(
    handle=DATASET_HANDLE,
    local_dataset_dir=str(SOURCE_DIR),
    version_notes=VERSION_NOTES,
    ignore_patterns=ignore_patterns,
)

Uploading Dataset https://kaggle.com/datasets/wwip62/nlp-hw2-runs ...
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/config.yaml
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/metrics/examples_history.json
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/metrics/epochs_history.json
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/checkpoints/last.pt
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/checkpoints/best.pt
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/results/summary.json
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/init_state/sample_indices.json
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/

Uploading:   0%|          | 0.00/1.13M [00:00<?, ?B/s]















Uploading:   1%|▏         | 16.4k/1.13M [00:00<00:12, 88.2kB/s]


Uploading:  17%|█▋        | 197k/1.13M [00:00<00:01, 715kB/s]  











Uploading:  57%|█████▋    | 639k/1.13M [00:00<00:00, 1.13MB/s]





Uploading:  83%|████████▎ | 934k/1.13M [00:00<00:00, 1.48MB/s]



Uploading: 100%|██████████| 923/923 [00:00<00:00, 1.15kB/s]
Uploading: 100%|██████████| 1.83k/1.83k [00:00<00:00, 2.48kB/s]

Upload successful: /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/config.yaml (923B)
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/init_state/wandb.json
Upload successful: /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/results/summary.json (2KB)
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/wandb/run-20261022_222020-2424xg38/run-2424xg38.wandb







Uploading: 100%|██████████| 1.06k/1.06k [00:00<00:00, 1.15kB/s]

Upload successful: /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/metrics/epochs_history.json (1KB)
Starting upload for file /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/wandb/run-20261022_222020-2424xg38/logs/debug-internal.log


...

Upload successful: /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/wandb/run-20261022_222020-2424xg38/logs/debug-internal.log (542KB)













Uploading: 100%|██████████| 1.34k/1.34k [00:01<00:00, 911B/s]  

Upload successful: /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/wandb/run-20261022_222020-2424xg38/files/wandb-metadata.json (1KB)







































Uploading: 100%|██████████| 4.69M/4.69M [00:06<00:00, 738kB/s] 

Upload successful: /Users/maks62wip/ML/NLP/HW-2/experiment_runs/baseline_plus_plus_longer/wandb/run-20261022_222020-2424xg38/run-2424xg38.wandb (4MB)


## Как восстановить experiment_runs на Kaggle

Этот фрагмент надо вставить в ноутбук **на Kaggle**, после клонирования репозитория в `/kaggle/working` и настройки импортов. Укажи тот же `DATASET_HANDLE`.

Файлы копируются из подключённого датасета в привычную папку проекта; это сохраняет пути вида `PROJECT_DIR / "experiment_runs" / имя_эксперимента`.

```python
import shutil
from pathlib import Path
import kagglehub

from lib.utils.paths import PROJECT_DIR

DATASET_HANDLE = "YOUR_USERNAME/nlp-hw2-runs"
dataset_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))
shutil.copytree(
    dataset_dir,
    PROJECT_DIR / "experiment_runs",
    dirs_exist_ok=True,
)
```

Стоит выполнить подключение датасета в интерактивной сессии перед `Save Version`. Для фиксации версии можно использовать handle вида `username/nlp-hw2-runs/versions/1`.